In [0]:
%sql
select current_metastore(), current_catalog(), current_user();


show catalogs;

In [0]:
%sql
-- Create the main ShopSphere catalog
CREATE CATALOG IF NOT EXISTS shopsphere_catalog
  COMMENT 'ShopSphere Analytics Central Data Lakehouse Catalog. 
           Contains all retail, marketing, and risk domain data.';
-- Verify creation
DESCRIBE CATALOG shopsphere_catalog;


In [0]:
%sql
-- Set active catalog first (avoids typing catalog name every time)
USE CATALOG shopsphere_catalog;
-- Create domain schemas
CREATE SCHEMA IF NOT EXISTS retail
  COMMENT 'ShopSphere Retail Domain: orders, products, sellers, payments.';
CREATE SCHEMA IF NOT EXISTS marketing
  COMMENT 'ShopSphere Marketing Domain: customer segments, campaigns, reviews.';
CREATE SCHEMA IF NOT EXISTS risk
  COMMENT 'ShopSphere Risk & Compliance Domain: audit data, access logs, fraud signals.';
-- Verify all schemas exist
SHOW SCHEMAS IN shopsphere_catalog;


In [0]:
%sql
-- View the current managed location for the catalog
DESCRIBE CATALOG EXTENDED shopsphere_catalog;
-- Verify
DESCRIBE SCHEMA EXTENDED shopsphere_catalog.retail;


In [0]:
spark.sql("USE CATALOG shopsphere_catalog")
spark.sql("USE SCHEMA retail")


In [0]:
orders_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .load("/Volumes/shopsphere_catalog/retail/raw_ingestion/olist_orders_dataset.csv")
# copy the path from your Databricks catalog
# Inspect schema
orders_raw.printSchema()
print(f"Row count: {orders_raw.count():,}")


In [0]:
from pyspark.sql.functions import col, to_timestamp, when, upper
orders_clean = orders_raw \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("order_delivered_customer_date",
                to_timestamp(col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("order_estimated_delivery_date",
                to_timestamp(col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("order_status", upper(col("order_status")))


In [0]:
orders_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("shopsphere_catalog.retail.orders")

print("Orders table created successfully!")


In [0]:
order_items = spark.read.format("csv") \
    .option("header", "true").option("inferSchema", "true") \
    .load("/Volumes/shopsphere_catalog/retail/raw_ingestion/olist_order_items_dataset.csv")
order_items.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("shopsphere_catalog.retail.order_items")


In [0]:
customers = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/shopsphere_catalog/retail/raw_ingestion/olist_customers_dataset.csv")
customers.printSchema()
print("Row count:", customers.count())
customers.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("shopsphere_catalog.retail.customers")


In [0]:
# Products table
products = spark.read.format("csv") \
    .option("header", "true").option("inferSchema", "true") \
    .load("/Volumes/shopsphere_catalog/retail/raw_ingestion/olist_products_dataset.csv")
products.printSchema()
print("Row count:", products.count())
products.write.format("delta").mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("shopsphere_catalog.retail.products")


In [0]:
# Sellers table
sellers = spark.read.format("csv") \
    .option("header", "true").option("inferSchema", "true") \
    .load("/Volumes/shopsphere_catalog/retail/raw_ingestion/olist_sellers_dataset.csv")
sellers.printSchema()
print("Row count:", sellers.count())
sellers.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("shopsphere_catalog.retail.sellers")


In [0]:
print("All managed tables created!")
spark.sql("SHOW TABLES IN shopsphere_catalog.retail").show()


In [0]:
%sql
-- Create Target Volume (if not exists)
CREATE VOLUME IF NOT EXISTS shopsphere_catalog.retail.external_data;


In [0]:
dbutils.fs.cp(
    "/Volumes/shopsphere_catalog/retail/raw_ingestion/olist_geolocation_dataset.csv",
    "/Volumes/shopsphere_catalog/retail/external_data/olist_geolocation_dataset.csv"
)


In [0]:
display(dbutils.fs.ls("/Volumes/shopsphere_catalog/retail/external_data/"))

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/shopsphere_catalog/retail/external_data/olist_geolocation_dataset.csv")
df.printSchema()
print("Row count:", df.count())
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("geolocation_ext")


In [0]:
%sql
SHOW TABLES IN shopsphere_catalog.retail;


In [0]:
%sql
-- Switch to SQL Editor for cleaner output
USE CATALOG shopsphere_catalog;
USE SCHEMA retail;

-- Row count validation
SELECT 'orders'     AS table_name, COUNT(*) AS rows FROM orders
UNION ALL
SELECT 'customers',  COUNT(*) FROM customers
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'products',   COUNT(*) FROM products
UNION ALL
SELECT 'sellers',    COUNT(*) FROM sellers;

-- Business metric: Orders by status
SELECT order_status, COUNT(*) AS order_count,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;

-- Join test: Top 5 revenue-generating product categories
SELECT p.product_category_name, 
       SUM(oi.price) AS total_revenue,
       COUNT(DISTINCT oi.order_id) AS order_count
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY total_revenue DESC
LIMIT 10;


In [0]:
%sql
-- Grant full access to data engineers on the entire catalog
GRANT USE CATALOG ON CATALOG shopsphere_catalog TO shopsphere_data_engineers;


In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA shopsphere_catalog.retail TO shopsphere_data_engineers;


In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA shopsphere_catalog.marketing TO shopsphere_data_engineers;


In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA shopsphere_catalog.risk TO shopsphere_data_engineers;


In [0]:
%sql
GRANT USE SCHEMA, CREATE TABLE 
ON SCHEMA shopsphere_catalog.retail 
TO shopsphere_data_engineers;
GRANT SELECT, MODIFY
ON SCHEMA shopsphere_catalog.retail
TO shopsphere_data_engineers;


In [0]:
%sql
GRANT USE SCHEMA, CREATE TABLE
ON SCHEMA shopsphere_catalog.marketing
TO shopsphere_data_engineers;

GRANT SELECT, MODIFY
ON SCHEMA shopsphere_catalog.marketing
TO shopsphere_data_engineers;


In [0]:
%sql
GRANT USE SCHEMA, CREATE TABLE
ON SCHEMA shopsphere_catalog.risk
TO shopsphere_data_engineers;

GRANT SELECT, MODIFY
ON SCHEMA shopsphere_catalog.risk
TO shopsphere_data_engineers;


In [0]:
%sql
-- Verify
SHOW GRANTS ON CATALOG shopsphere_catalog;


In [0]:
%sql
-- Marketing Analysts: USE CATALOG (required) + READ-ONLY on retail + marketing
GRANT USE CATALOG ON CATALOG shopsphere_catalog TO shopsphere_marketing_analysts;
GRANT USE SCHEMA ON SCHEMA shopsphere_catalog.retail TO shopsphere_marketing_analysts;
GRANT SELECT ON TABLE shopsphere_catalog.retail.orders TO shopsphere_marketing_analysts;
GRANT SELECT ON TABLE shopsphere_catalog.retail.order_items TO shopsphere_marketing_analysts;


In [0]:
%sql
GRANT SELECT ON TABLE shopsphere_catalog.retail.products TO shopsphere_marketing_analysts;
-- NOTE: NO access granted to customers (has zip_code PII) or risk schema

GRANT USE SCHEMA ON SCHEMA shopsphere_catalog.marketing TO shopsphere_marketing_analysts;
GRANT SELECT ON SCHEMA shopsphere_catalog.marketing TO shopsphere_marketing_analysts;

-- Test: what can marketing analysts see?
SHOW GRANTS ON TABLE shopsphere_catalog.retail.orders;


In [0]:
%sql
-- Create a safe view with PII masking
CREATE OR REPLACE VIEW shopsphere_catalog.retail.customers_masked AS
SELECT 
    customer_id,
    CONCAT(LEFT(customer_zip_code_prefix, 3), '**') AS customer_zip_masked,  -- mask last 2 digits
    customer_city,
    customer_state,
    -- Tag this view
    'masked' AS pii_level
FROM shopsphere_catalog.retail.customers;
-- Grant marketing analysts access to the MASKED view only
GRANT SELECT ON VIEW shopsphere_catalog.retail.customers_masked TO shopsphere_marketing_analysts;
-- Verify the masking works
SELECT * FROM shopsphere_catalog.retail.customers_masked LIMIT 10;


In [0]:
%sql
-- Audit: All grants on the retail schema
SHOW GRANTS ON SCHEMA shopsphere_catalog.retail; 


In [0]:
%sql
-- Audit: Specific table grants
SHOW GRANTS ON TABLE shopsphere_catalog.retail.customers;
SHOW GRANTS ON VIEW shopsphere_catalog.retail.customers_masked;


In [0]:
%sql
-- Enable system schemas (Admin step)
SHOW SCHEMAS IN system;


In [0]:
%sql
SELECT * FROM system.access.audit LIMIT 5;


In [0]:
%sql
DESCRIBE TABLE system.access.audit;


In [0]:
%sql
-- Summary: Access count per user
SELECT
    user_identity.email AS user_email,
    COUNT(*) AS access_count,
    COUNT(DISTINCT event_date) AS days_accessed,
    MIN(event_time) AS first_access,
    MAX(event_time) AS last_access
FROM system.access.audit
WHERE
    event_date >= CURRENT_DATE - 30
    AND request_params['table_full_name'] LIKE '%shopsphere%'
    AND response.status_code = 200
GROUP BY user_identity.email
ORDER BY access_count DESC;


In [0]:
%sql
-- DDL Audit: All schema changes in last 30 days
SELECT 
    event_time,
    user_identity.email AS changed_by,
    action_name AS ddl_action,
    request_params['catalog_name'] AS catalog,
    request_params['schema_name'] AS schema_name,
    request_params['table_name'] AS table_name,
    response.status_code AS result,
    response.error_message AS error
FROM system.access.audit WHERE 
    event_date >= CURRENT_DATE - 30
    AND service_name = 'unityCatalog'
    AND action_name IN (
        'createTable', 'deleteTable', 'updateTable', 
        'createSchema', 'deleteSchema',
        'createCatalog', 'deleteCatalog',
        'updatePermissions'
    )
ORDER BY event_time DESC;
-- Failed access attempts (security monitoring)
SELECT
    event_time,
    user_identity.email AS attempted_by,
    action_name,
    response.error_message AS denial_reason,
    source_ip_address
FROM system.access.audit
WHERE event_date >= CURRENT_DATE - 7 AND response.status_code != 200
ORDER BY event_time DESC LIMIT 50;


In [0]:
%sql
-- Create a compliance view in the risk schema
USE CATALOG shopsphere_catalog;
USE SCHEMA risk;

CREATE OR REPLACE VIEW shopsphere_catalog.risk.v_pii_access_log AS
SELECT
  CAST(event_time AS DATE) AS access_date,
  event_time,
  user_identity.email AS accessed_by,
  action_name,
  request_params['table_full_name'] AS table_name,
  source_ip_address AS client_ip,
  response.status_code AS outcome,
  CASE 
    WHEN response.status_code = 200 THEN 'yes' 
    ELSE 'no' 
  END AS status_icon
FROM system.access.audit
WHERE
  service_name = 'unityCatalog'
  AND (
    request_params['table_full_name'] LIKE '%customers%'
    OR request_params['table_full_name'] LIKE '%risk%'
  );
-- Grant risk team access to this view
GRANT SELECT ON VIEW shopsphere_catalog.risk.v_pii_access_log 
TO shopsphere_risk_team;
-- Test the view
SELECT * 
FROM shopsphere_catalog.risk.v_pii_access_log
ORDER BY event_time DESC
LIMIT 20;
